In [0]:
%sql
USE CATALOG heliosgrid_catalog;

CREATE SCHEMA IF NOT EXISTS gold
MANAGED LOCATION "s3://heliosgrid/catalog/gold/tables/";

In [0]:
%sql
CREATE OR REPLACE TABLE gold.dim_location AS
SELECT DISTINCT(station_id), station_name, state, climate_zone, latitude, longitude, elevation
FROM silver.cleaned_solar_telemetry;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold.dim_time AS
SELECT DISTINCT(record_timestamp) AS record_timestamp,
    DATE(record_timestamp) AS date_id,
    EXTRACT(YEAR FROM record_timestamp) AS year,
    EXTRACT(MONTH FROM record_timestamp) AS month,
    EXTRACT(DAY FROM record_timestamp) AS day,
    EXTRACT(DAYOFWEEK FROM record_timestamp) AS dayofweek,
    EXTRACT(HOUR FROM record_timestamp) AS hour,
    EXTRACT(MINUTE FROM record_timestamp) AS minute,
    monsoon_season,
    CASE WHEN ghi_wm2 > 0 THEN True ELSE False END AS is_daylight
FROM silver.cleaned_solar_telemetry;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold.dim_weather_condition AS
SELECT DISTINCT 
    MD5(CONCAT_WS('||', temp_celsius::CHAR(10), humidity_pct::CHAR(10), cloud_cover_pct::CHAR(10), aod::CHAR(10), pm2_5::CHAR(10))) AS weather_id,
    temp_celsius, humidity_pct, cloud_cover_pct, wind_speed_ms, aod, pm2_5, pm10, dust,
    CASE 
        WHEN pm2_5 <= 30 THEN 'Good'
        WHEN pm2_5 <= 60 THEN 'Moderate'
        WHEN pm2_5 <= 120 THEN 'Unhealthy'
        ELSE 'Severe Smog'
    END AS aq_category
FROM silver.cleaned_solar_telemetry;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold.fact_solar_generation AS
SELECT 
    MD5(CONCAT_WS('||', record_timestamp::CHAR(10), station_id::CHAR(10))) AS telemetry_id,
    record_timestamp, 
    station_id,
    MD5(CONCAT_WS('||', temp_celsius::CHAR(10), humidity_pct::CHAR(10), cloud_cover_pct::CHAR(10), aod::CHAR(10), pm2_5::CHAR(10))) AS weather_id,
    ghi_wm2, 
    dni_wm2, 
    dhi_wm2, 
    cell_temp_celsius, 
    thermal_derating_factor,
    ROUND((ghi_wm2 / 1000.0) * 100000.0 * thermal_derating_factor, 4) AS expected_power_kw,
    ghi_wm2 * thermal_derating_factor AS effective_irradiance_wm2
FROM silver.cleaned_solar_telemetry;

num_affected_rows,num_inserted_rows


In [0]:
# from pyspark.sql import functions as F

# # 1. Read Silver Table & Dimension Tables
# df_silver = spark.read.table("heliosgrid_catalog.silver.cleaned_solar_telemetry")
# df_dim_weather = spark.read.table("heliosgrid_catalog.gold.dim_weather_condition")

# # 2. Construct df_fact_new
# df_fact_new = (
#     df_silver
#     # A. Derive Primary & Foreign Keys
#     .withColumn("station_id", F.col("station_id"))
    
#     # B. Join with DIM_WEATHER to fetch weather_id
#     .join(
#         df_dim_weather,
#         on=["temp_celsius", "humidity_pct", "cloud_cover_pct", "wind_speed_ms", "aod", "pm2_5", "pm10", "dust"],
#         how="left"
#     )
    
#     # C. Calculate Fact Measures
#     .withColumn("expected_power_kw", (F.col("ghi_wm2") / 1000.0) * 100000.0 * F.col("thermal_derating_factor"))
#     .withColumn("effective_irradiance_wm2", F.col("ghi_wm2") * F.col("thermal_derating_factor"))
#     .withColumn("telemetry_id", F.md5(F.concat_ws("_", F.col("station_id"), F.col("record_timestamp"))))
    
#     # D. Select final Fact table schema
#     .select(
#         "telemetry_id",
#         "station_id",
#         "record_timestamp",
#         "weather_id",
#         "ghi_wm2",
#         "dni_wm2",
#         "dhi_wm2",
#         "cell_temp_celsius",
#         "thermal_derating_factor",
#         "expected_power_kw",
#         "effective_irradiance_wm2"
#     )
# )

In [0]:
df_fact_new = spark.sql("""
SELECT 
    MD5(CONCAT_WS('||', record_timestamp::CHAR(10), station_id::CHAR(10))) AS telemetry_id,
    record_timestamp, 
    station_id,
    MD5(CONCAT_WS('||', temp_celsius::CHAR(10), humidity_pct::CHAR(10), cloud_cover_pct::CHAR(10), aod::CHAR(10), pm2_5::CHAR(10))) AS weather_id,
    ghi_wm2, 
    dni_wm2, 
    dhi_wm2, 
    cell_temp_celsius, 
    thermal_derating_factor,
    ROUND((ghi_wm2 / 1000.0) * 100000.0 * thermal_derating_factor, 4) AS expected_power_kw,
    ghi_wm2 * thermal_derating_factor AS effective_irradiance_wm2
FROM silver.cleaned_solar_telemetry;""")

In [0]:
# 1. Deduplicate source rows on primary merge keys:
df_fact_deduped = df_fact_new.dropDuplicates(["telemetry_id"])

# 2. Register deduplicated DataFrame as staging_fact_view:
df_fact_deduped.createOrReplaceTempView("staging_fact_view")


In [0]:
%sql
MERGE INTO heliosgrid_catalog.gold.fact_solar_generation AS target
USING staging_fact_view AS source
ON target.station_id = source.station_id AND target.record_timestamp = source.record_timestamp
WHEN MATCHED THEN
  UPDATE SET 
    target.ghi_wm2 = source.ghi_wm2,
    target.dni_wm2 = source.dni_wm2,
    target.dhi_wm2 = source.dhi_wm2,
    target.cell_temp_celsius = source.cell_temp_celsius,
    target.thermal_derating_factor = source.thermal_derating_factor,
    target.expected_power_kw = source.expected_power_kw,
    target.effective_irradiance_wm2 = source.effective_irradiance_wm2
WHEN NOT MATCHED THEN
  INSERT (telemetry_id, station_id, record_timestamp, weather_id, ghi_wm2, dni_wm2, dhi_wm2, cell_temp_celsius, thermal_derating_factor, expected_power_kw, effective_irradiance_wm2)
  VALUES (source.telemetry_id, source.station_id, source.record_timestamp, source.weather_id, source.ghi_wm2, source.dni_wm2, source.dhi_wm2, source.cell_temp_celsius, source.thermal_derating_factor, source.expected_power_kw, source.effective_irradiance_wm2);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1056,1056,0,0
